In [4]:
import os
print(os.getcwd())

C:\Users\rosha\Documents\Projects\UWPhotonics\prob_tcn_for_LED


In [2]:
import subprocess
import sys
from pathlib import Path
import torch
import json
from Execute_channel import Execute_Channel
from experiments.QAT_experiment import read_model
from experiments.generate_Q88_time_frames import writeSentTime
from experiments.Plot_sv_constellation import create_plots
from experiments.Plot_EVM_vs_BitWidth import plot_evm_vs_bitwidth


In [5]:
# sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
# os.chdir(Path(__file__).resolve().parents[1])
# base_pth = os.path.abspath(os.path.join(os.path.dirname(__file__), "..", ".."))

current_dir = Path.cwd()
sys.path.insert(0, str(current_dir))
os.chdir(current_dir)
base_pth = current_dir
print(current_dir)

sim_directory = "sv_tcn/tcn6"
DATA_WIDTH = 16
TEST = 20
SAVE_PATH = f"{sim_directory}"
ED_MODEL = "data/experiments/test_real_gridsearch/encoder_decoder_20260825_1207/runs/tcn_ae_6f25ae0d"
CHANNEL_PTH = "data/experiments/test_real_gridsearch/channel_models_20260825_1204/runs/tcn_b338d2dc"

C:\Users\rosha\Documents\Projects\UWPhotonics\prob_tcn_for_LED


In [12]:
writeSentTime(SAVE_PATH, DATA_WIDTH)
read_model(os.path.join(base_pth, ED_MODEL, "model.pt"), os.path.join(base_pth, SAVE_PATH, "TestingData", f"Test{TEST}"), DATA_WIDTH)

  Saved 3760 values → C:\Users\rosha\Documents\Projects\UWPhotonics\sv_tcn/tcn6\input_time_series.mem


In [ ]:
result = subprocess.run(
        ["iverilog", "-g2012", "-P", f'tb.MODEL_TYPE="encoder"', "-P", f"tb.TEST={TEST}", "-P", f"tb.DATA_WIDTH={DATA_WIDTH}", "-P", f"tb.SAMPLES={3760}", "-o", "tb.vvp", "tb.sv"],
        cwd=sim_directory,
        capture_output=True,
        text=True,
        check=True,
    )

result2 = subprocess.run(
    ["vvp", "tb.vvp"],
    cwd=sim_directory,
    capture_output=True,
    text=True,
    check=True,
)

print(result2.stdout)

In [ ]:
Execute_Channel(DATA_WIDTH, CHANNEL_PTH, SAVE_PATH)

In [6]:
result3 = subprocess.run(
        ["iverilog", "-g2012", "-P", f'tb.MODEL_TYPE="decoder"', "-P", f"tb.TEST={TEST}", "-P", f"tb.DATA_WIDTH={DATA_WIDTH}", "-P", f"tb.SAMPLES={3760}", "-o", "tb.vvp", "tb.sv"],
        cwd=sim_directory,
        capture_output=True,
        text=True,
        check=True,
    )

result4 = subprocess.run(
    ["vvp", "tb.vvp"],
    cwd=sim_directory,
    capture_output=True,
    text=True,
    check=True,
)

print(result4.stdout)

Network Latency: 36
DATAWIDTH: 16, MODELTYPE:decoder, TEST:20
VCD info: dumpfile tb.vcd opened for output.
tb.sv:81: $finish called at 23076 (1s)



In [ ]:
create_plots(DATA_WIDTH, ShowTimeSeries=False, ed_model_pth=os.path.join(base_pth, ED_MODEL))